# HQViT — Clean Restart Notebook
**Data-Efficient Image Classification Using a Hybrid Quantum-Classical Vision Transformer**  
CIFAR-10 · CPU-first · maximum 15 epochs.

This is a clean, linear replacement for the fragmented notebook. It preserves the original split procedure from Cells 12–13. It does not overwrite historical results or claim that the rebuilt HQViT is identical to the lost historical model/checkpoint.

**Important:** The original split seed must be recovered from `results/experiment_config.json` or entered explicitly. The notebook stops rather than guessing. The test set is reserved for final evaluation. Training is opt-in.

## 1. Imports and environment

In [ ]:
import os, sys, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import pennylane as qml
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")
DEVICE = torch.device("cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PennyLane:", qml.__version__)
print("Device:", DEVICE)

## 2. Configuration and seed recovery

In [ ]:
CONFIG_PATH = Path("results/experiment_config.json")
ORIGINAL_SEED = None  # If config is missing, enter the exact original SEED here.

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        saved_config = json.load(f)
    if "seed" in saved_config:
        ORIGINAL_SEED = int(saved_config["seed"])

if ORIGINAL_SEED is None:
    raise RuntimeError(
        "Original seed not found. Set ORIGINAL_SEED to the exact SEED "
        "from the original notebook. Do not guess."
    )

SEED = ORIGINAL_SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BATCH_SIZE, MAX_EPOCHS = 32, 15
LEARNING_RATE, WEIGHT_DECAY = 1e-4, 1e-4
DATA_FRACTIONS = (0.10, 0.25, 0.50, 1.00)
DATA_DIR = Path("data")
RESULTS_DIR = Path("results_clean")
CHECKPOINT_DIR = Path("checkpoints_clean")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
print("Original seed:", SEED)

## 3. CIFAR-10 and original split reconstruction

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

full_train_dataset = datasets.CIFAR10(
    root=str(DATA_DIR), train=True, download=True, transform=transform
)
test_dataset = datasets.CIFAR10(
    root=str(DATA_DIR), train=False, download=True, transform=transform
)
assert len(full_train_dataset) == 50000 and len(test_dataset) == 10000

# Original Cell 12: PyTorch random_split with seeded Generator.
split_generator = torch.Generator().manual_seed(SEED)
train_pool, validation_dataset = random_split(
    full_train_dataset, [45000, 5000], generator=split_generator
)

# Original Cell 13: independently seeded NumPy permutation of train_pool.
pool_order = np.random.default_rng(SEED).permutation(len(train_pool))
train_subsets = {
    f: Subset(train_pool, pool_order[:int(len(train_pool)*f)].tolist())
    for f in DATA_FRACTIONS
}
train_indices = list(train_pool.indices)
validation_indices = list(validation_dataset.indices)
assert not (set(train_indices) & set(validation_indices))
assert len(set(train_indices + validation_indices)) == 50000

print("Train pool:", len(train_pool), "| Validation:", len(validation_dataset),
      "| Test:", len(test_dataset))
for f, ds in train_subsets.items(): print(f"{f:.0%}:", len(ds))

## 4. Loaders and sanity check

In [ ]:
def make_loaders(fraction=1.0, batch_size=BATCH_SIZE):
    tr = DataLoader(train_subsets[fraction], batch_size=batch_size,
                    shuffle=True, num_workers=0)
    va = DataLoader(validation_dataset, batch_size=batch_size,
                    shuffle=False, num_workers=0)
    return tr, va

train_loader, validation_loader = make_loaders(0.10)
x, y = next(iter(train_loader))
print("Batch:", tuple(x.shape), tuple(y.shape),
      "| finite:", bool(torch.isfinite(x).all()))
assert x.shape[1:] == (3, 32, 32) and torch.isfinite(x).all()

## 5. Model definitions

The comparison consists of a Classical ViT, a classical 4-dimensional bottleneck control, and a rebuilt 4-qubit HQViT. All are freshly initialized. The HQViT is the documented rebuilt implementation, **not** a reproduction of the missing historical weights.

In [ ]:
class ViTBackbone(nn.Module):
    def __init__(self, dim=128, heads=4, layers=4, patch=4, dropout=0.1):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 65, dim))
        self.dropout = nn.Dropout(dropout)
        enc = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=4*dim,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=layers)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = self.dropout(torch.cat([cls, x], dim=1) + self.pos_embed)
        return self.transformer(x)[:, 0]

class ClassicalViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTBackbone()
        self.head = nn.Linear(128, 10)
    def forward(self, x): return self.head(self.vit(x))

class ClassicalBottleneck(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTBackbone()
        self.projection_norm = nn.LayerNorm(128)
        self.projection = nn.Linear(128, 4)
        self.classical_transform = nn.Sequential(nn.Linear(4, 4), nn.Tanh())
        self.classifier = nn.Sequential(nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 10))
    def forward(self, x):
        z = self.projection(self.projection_norm(self.vit(x)))
        return self.classifier(self.classical_transform(z))

N_QUBITS, Q_LAYERS = 4, 2
qdev = qml.device("default.qubit", wires=N_QUBITS, shots=None)

@qml.qnode(qdev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for w in range(N_QUBITS): qml.RY(inputs[w], wires=w)
    for layer in range(Q_LAYERS):
        for w in range(N_QUBITS):
            qml.RY(weights[layer, w, 0], wires=w)
            qml.RZ(weights[layer, w, 1], wires=w)
        for w in range(N_QUBITS-1): qml.CNOT(wires=[w, w+1])
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

class HQViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTBackbone()
        self.projection_norm = nn.LayerNorm(128)
        self.projection = nn.Linear(128, N_QUBITS)
        self.quantum_weights = nn.Parameter(0.01 * torch.randn(Q_LAYERS, N_QUBITS, 2))
        self.classifier = nn.Sequential(nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 10))
    def forward(self, x):
        z = self.projection(self.projection_norm(self.vit(x)))
        q = torch.stack([torch.stack(quantum_circuit(row, self.quantum_weights)) for row in z])
        return self.classifier(q.to(dtype=z.dtype, device=z.device))

def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for cls in (ClassicalViT, ClassicalBottleneck, HQViT):
    m = cls()
    print(cls.__name__, parameter_count(m), "trainable parameters")
    del m

## 6. Smoke test (forward + backward)

In [ ]:
# Tiny batch only; validates shapes and gradient flow, not model quality.
sx, sy = next(iter(DataLoader(Subset(train_subsets[0.10], range(4)),
                              batch_size=4, shuffle=False)))
for cls in (ClassicalViT, ClassicalBottleneck, HQViT):
    model = cls().to(DEVICE)
    model.zero_grad(set_to_none=True)
    logits = model(sx)
    loss = nn.CrossEntropyLoss()(logits, sy)
    loss.backward()
    finite_grads = all(p.grad is None or torch.isfinite(p.grad).all()
                       for p in model.parameters())
    print(cls.__name__, logits.shape, "finite output:", bool(torch.isfinite(logits).all()),
          "finite gradients:", bool(finite_grads))
    assert logits.shape == (4, 10) and torch.isfinite(logits).all() and finite_grads
    del model

## 7. Training utilities

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            yb = yb.to(DEVICE)
            loss_sum += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum/total, correct/total

def train_model(model, name, fraction=0.10, epochs=1):
    epochs = min(int(epochs), MAX_EPOCHS)
    tr_loader, va_loader = make_loaders(fraction)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    rows, best_acc = [], -1.0
    ckpt = CHECKPOINT_DIR / f"{name}_{int(fraction*100)}pct_best.pt"

    for epoch in range(1, epochs+1):
        start = time.time()
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()*len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
        val_loss, val_acc = evaluate(model, va_loader)
        row = dict(model=name, fraction=fraction, epoch=epoch,
                   train_loss=loss_sum/total, train_accuracy=correct/total,
                   validation_loss=val_loss, validation_accuracy=val_acc,
                   epoch_seconds=time.time()-start)
        rows.append(row)
        pd.DataFrame(rows).to_csv(RESULTS_DIR/f"{name}_{int(fraction*100)}pct.csv",
                                  index=False)
        print(f"{name} {epoch}/{epochs} train={correct/total:.4f} "
              f"val={val_acc:.4f} time={row['epoch_seconds']:.1f}s")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({"model_state_dict": model.state_dict(), **row, "seed": SEED}, ckpt)
    return model, pd.DataFrame(rows), ckpt

## 8. Run experiments — deliberately disabled by default

The historical Classical ViT completed 15 epochs: best validation accuracy **68.16% at epoch 12**, final validation accuracy **67.88%**. Its 71.54% figure was training accuracy, not validation accuracy. Preserve those results; do not retrain or overwrite them by default.

The HQViT evaluates a quantum circuit per sample and may be very slow on CPU. Begin with one epoch on 10% data.

In [ ]:
RUN_EXPERIMENTS = False
RUN_FRACTION = 0.10
RUN_EPOCHS = 1  # never more than 15

if RUN_EXPERIMENTS:
    # Uncomment exactly ONE model run at a time.
    # m, hist, path = train_model(ClassicalBottleneck(), "classical_bottleneck",
    #                             RUN_FRACTION, RUN_EPOCHS)
    # m, hist, path = train_model(HQViT(), "rebuilt_hqvit",
    #                             RUN_FRACTION, RUN_EPOCHS)
    print("Activate one explicit train_model(...) call above.")
else:
    print("Training disabled. Setup and smoke tests only.")

## 9. Load checkpoints

In [ ]:
def load_checkpoint(model, path):
    saved = torch.load(path, map_location=DEVICE)
    model.load_state_dict(saved["model_state_dict"])
    model.to(DEVICE).eval()
    print("Loaded:", path)
    print("Metadata:", {k:v for k,v in saved.items() if k != "model_state_dict"})
    return model

# Example:
# selected_model = load_checkpoint(ClassicalViT(),
#     "models_clean/section94/classical_vit_best.pt")

## 10. Final test evaluation — only after model selection

In [ ]:
def final_test_evaluation(model):
    # Do not use test metrics to tune or select models.
    loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0)
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(yb.numpy().tolist())
    print("Test accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))
    return np.asarray(y_true), np.asarray(y_pred)

# y_test, predictions = final_test_evaluation(selected_model)

## 11. Save experiment configuration

In [ ]:
record = {
    "project": "Data-Efficient Image Classification Using a Hybrid Quantum-Classical Vision Transformer",
    "dataset": "CIFAR-10",
    "seed": SEED,
    "split": "torch.random_split([45000, 5000], generator=torch.Generator().manual_seed(SEED))",
    "training_pool": len(train_pool),
    "validation": len(validation_dataset),
    "test": len(test_dataset),
    "fractions": list(DATA_FRACTIONS),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "device": str(DEVICE),
    "hqvit": "fresh rebuilt implementation; not asserted identical to historical model"
}
with open(RESULTS_DIR/"clean_experiment_config.json", "w") as f:
    json.dump(record, f, indent=2)
print(json.dumps(record, indent=2))
print("Clean restart notebook setup complete.")